# Identificador de imágenes

In [2]:
import tensorflow as tf
print(tf.__version__)

2.21.0


In [3]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [4]:
# 1. Crear el generador con normalización
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2) # Reservamos 20% para test

# 2. Cargar y estandarizar imágenes de entrenamiento
trdata = datagen.flow_from_directory(
    directory='/workspaces/IgnacioSabinoG-IntroML/data/raw/dogs-vs-cats/train',
    target_size=(224, 224),  # Aquí se cumple la estandarización de tamaño
    batch_size=8,
    class_mode='categorical',
    subset='training'
)

# 3. Cargar y estandarizar imágenes de prueba (validación)
tsdata = datagen.flow_from_directory(
    directory='/workspaces/IgnacioSabinoG-IntroML/data/raw/dogs-vs-cats/train',
    target_size=(224, 224),
    batch_size=8,
    class_mode='categorical',
    subset='validation'
)

Found 20000 images belonging to 2 classes.
Found 5000 images belonging to 2 classes.


In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPool2D, Flatten, Dense
from tensorflow.keras.optimizers import Adam

# 1. Definir la arquitectura VGG16 (apilando capas convolucionales y de pooling)
model = Sequential([
    # Bloque 1
    Conv2D(64, (3,3), padding="same", activation="relu", input_shape=(224,224,3)),
    Conv2D(64, (3,3), padding="same", activation="relu"),
    MaxPool2D((2,2), strides=(2,2)),
    # Bloque 2
    Conv2D(128, (3,3), padding="same", activation="relu"),
    Conv2D(128, (3,3), padding="same", activation="relu"),
    MaxPool2D((2,2), strides=(2,2)),
    # Bloque 3
    Conv2D(256, (3,3), padding="same", activation="relu"),
    Conv2D(256, (3,3), padding="same", activation="relu"),
    Conv2D(256, (3,3), padding="same", activation="relu"),
    MaxPool2D((2,2), strides=(2,2)),
    # Bloque 4
    Conv2D(512, (3,3), padding="same", activation="relu"),
    Conv2D(512, (3,3), padding="same", activation="relu"),
    Conv2D(512, (3,3), padding="same", activation="relu"),
    MaxPool2D((2,2), strides=(2,2)),
    # Bloque 5
    Conv2D(512, (3,3), padding="same", activation="relu"),
    Conv2D(512, (3,3), padding="same", activation="relu"),
    Conv2D(512, (3,3), padding="same", activation="relu"),
    MaxPool2D((2,2), strides=(2,2)),
    # Capas densas (clasificador)
    Flatten(),
    Dense(256, activation="relu"), # Bajamos de 4096 a 256 para salvar la RAM
    Dense(256, activation="relu"), # Bajamos de 4096 a 256
    Dense(2, activation="softmax")
])

# 2. Compilar el modelo con Adam y pérdida categórica
model.compile(optimizer=Adam(learning_rate=0.0001), 
              loss='categorical_crossentropy', 
              metrics=['accuracy'])


In [6]:
from tensorflow.keras.optimizers import Adam

# 1. Compilar el modelo
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# 1. Definir los Callbacks (Paso 4)
checkpoint = ModelCheckpoint(
    "mejor_modelo_vgg16.keras", # Nombre del archivo donde se guardará
    monitor='val_accuracy', 
    save_best_only=True, 
    mode='max',
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=3, # Si en 3 épocas no mejora, se detiene
    verbose=1
)

# 2. Entrenar el modelo con los callbacks
history = model.fit(
    trdata,
    steps_per_epoch=100,      
    validation_data=tsdata,   # Añadimos los datos de validación
    validation_steps=50,      
    epochs=5,
    callbacks=[checkpoint, early_stop] 
    )

Epoch 1/5


I0000 00:00:1776533127.640481   40550 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.5504 - loss: 0.6928
Epoch 1: val_accuracy improved from None to 0.49500, saving model to mejor_modelo_vgg16.keras

Epoch 1: finished saving model to mejor_modelo_vgg16.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 781s 8s/step - accuracy: 0.5213 - loss: 0.6931 - val_accuracy: 0.4950 - val_loss: 0.6933
Epoch 2/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.5067 - loss: 0.6932
Epoch 2: val_accuracy improved from 0.49500 to 0.50250, saving model to mejor_modelo_vgg16.keras

Epoch 2: finished saving model to mejor_modelo_vgg16.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 775s 8s/step - accuracy: 0.5063 - loss: 0.6933 - val_accuracy: 0.5025 - val_loss: 0.6931
Epoch 3/5
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 7s/step - accuracy: 0.4903 - loss: 0.6934
Epoch 3: val_accuracy improved from 0.50250 to 0.51750, saving model to mejor_modelo_vgg16.keras

Epoch 3: finished saving model to mejor_modelo_vgg16.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 776s 8s/step - accuracy: 

In [7]:
from tensorflow.keras.models import load_model
import numpy as np

# 1. Cargar el mejor modelo que guardó el Checkpoint
model_opt = load_model('mejor_modelo_vgg16.keras')

# 2. Utilizar el conjunto de test para hacer predicciones
# Sacamos un lote (batch) de imágenes de prueba
test_images, test_labels = next(tsdata)

# Realizamos la predicción
predictions = model_opt.predict(test_images)

# Convertir las predicciones de probabilidades a etiquetas (0 o 1)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = np.argmax(test_labels, axis=1)

print(f"Predicciones para el batch: {predicted_classes}")
print(f"Etiquetas reales: {true_classes}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
Predicciones para el batch: [0 0 0 0 0 0 0 0]
Etiquetas reales: [1 0 1 1 0 1 1 0]


El modelo "no sabe" todavía: Al haber predicho [0 0 0 0 0 0 0 0] frente a una realidad mezclada ([1 0 1 1...]), significa que el modelo se ha sesgado hacia una sola clase (probablemente "gato"). Esto es normal porque solo ha visto una fracción mínima de las 25,000 imágenes y por muy pocas épocas.

In [8]:
import os
from tensorflow.keras.models import load_model

# 1. Definir la ruta de destino (siguiendo tu estructura)
save_path = '/workspaces/IgnacioSabinoG-IntroML/src/vgg16_cats_dogs.keras'

# 2. Guardar el modelo final entrenado
model.save(save_path)

print(f"✅ Modelo guardado exitosamente en: {save_path}")

# 3. Carga de prueba (Paso 4 - parte final)
# Esto demuestra que puedes recuperar el "mejor modelo" guardado por el checkpoint
modelo_final = load_model(save_path)
print("✅ Modelo cargado correctamente para predicciones.")

✅ Modelo guardado exitosamente en: /workspaces/IgnacioSabinoG-IntroML/src/vgg16_cats_dogs.keras
✅ Modelo cargado correctamente para predicciones.


In [ ]:
import os

# Buscamos cualquier archivo que termine en .keras en todo el espacio de trabajo
found = False
for root, dirs, files in os.walk('/workspaces/IgnacioSabinoG-IntroML/'):
    for file in files:
        if file.endswith(".keras"):
            full_path = os.path.join(root, file)
            size_mb = os.path.getsize(full_path) / (1024 * 1024)
            print(f"📍 Encontrado: {full_path}")
            print(f"⚖️  Tamaño: {size_mb:.2f} MB")
            found = True

if not found:
    print("❌ No se encontró ningún archivo .keras. Revisa si el model.save() se ejecutó correctamente.")

📍 Encontrado: /workspaces/IgnacioSabinoG-IntroML/src/mejor_modelo_vgg16.keras
⚖️  Tamaño: 242.76 MB
📍 Encontrado: /workspaces/IgnacioSabinoG-IntroML/src/vgg16_cats_dogs.keras
⚖️  Tamaño: 242.76 MB


: 